# Orang 3 - Evaluasi dan Metrik LLM

In [1]:
import os
import pandas as pd
import numpy as np
import time
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate


In [ ]:
os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY", "")

In [3]:
embed_model = HuggingFaceEmbeddings(
    model_name='paraphrase-multilingual-MiniLM-L12-v2'
)

vectorstore = Chroma(
    persist_directory="../artifacts/chroma",
    collection_name="banaspati",
    embedding_function=embed_model
)

In [34]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0.1
)

In [21]:
template = """Anda adalah BANASPATI (Bubur Panas Personal Assistant), asisten akademik yang ahli.
Tugas Anda adalah menjawab pertanyaan pengguna HANYA BERDASARKAN KONTEKS DOKUMEN yang diberikan di bawah ini.

ATURAN KETAT:
1. Baca dan pahami SELURUH konteks dokumen dengan teliti sebelum menjawab.
2. Jika jawabannya ADA di dalam konteks (meskipun tersebar di beberapa dokumen), rangkum dan jawab dengan lengkap.
3. Jika informasi BENAR-BENAR tidak ada di dalam konteks, jawab HANYA dengan: "informasi tidak ditemukan"
4. Jangan pernah mengarang jawaban atau menggunakan pengetahuan dari luar dokumen.

=== KONTEKS DOKUMEN ===
{context}
=== AKHIR KONTEKS ===

Pertanyaan: {question}

Jawaban:"""
prompt = PromptTemplate(template=template, input_variables=["context", "question"])

In [22]:
df = pd.read_csv("../data/banaspati_eval_questions.csv")
df.head()

,question_id,user_input,reference,expected_reference
0,Q01,"Pada hari Rabu, mahasiswa Semester 4 ingin men...","Ya, jadwalnya bentrok karena keduanya berlangs...",Jadwal Perkuliahan.docx; image1.png; bagian Ra...
1,Q02,"Pada hari Selasa pukul 13.00–15.30, ada kelas ...",Pada Selasa pukul 13.00–15.30 terdapat dua kel...,Jadwal Perkuliahan.docx; image2.png; bagian Se...
2,Q03,"Pada hari Kamis, kelas DTI apa saja yang dijad...",Pada Kamis di Lab 902 terdapat dua kelas DTI. ...,Jadwal Perkuliahan.docx; image1.png; bagian Ka...
3,Q04,"Berdasarkan kurikulum dan jadwal perkuliahan, ...","Ya. Dalam kurikulum, Kecerdasan Artifisial dan...",Kurikulum.pdf; Tabel 17 Daftar Mata Kuliah Sem...
4,Q05,Seorang mahasiswa aktif S1 semester 6 sudah lu...,Mahasiswa memenuhi syarat dasar jika merupakan...,Sosialisasi Magang dan Prestasi DTI.pdf; slide...


In [23]:
def banaspati_answer_eval(question):
    start_time = time.perf_counter()
    raw_docs = vectorstore.similarity_search(
        question,
        k=40
    )

    question_words = set(question.lower().split())

    stopwords = {
        'yang', 'di', 'dan', 'ini', 'itu',
        'ada', 'dalam', 'untuk', 'pada', 'ke', 'dengan',
        'adalah', 'apa', 'atau', 'sebutkan', 'jelaskan', 'beberapa',
        'bagaimana', 'berapa', 'apakah'
    }

    keywords = question_words - stopwords

    def score_doc(doc):
        text_lower = doc.page_content.lower()

        length_penalty = (
            0
            if len(doc.page_content) > 200
            else -5
        )
        keyword_hits = sum(
            1
            for kw in keywords
            if kw in text_lower
        )

        return keyword_hits + length_penalty

    docs = sorted(
        raw_docs,
        key=score_doc,
        reverse=True
    )[:10]
    contexts = [
        d.page_content
        for d in docs
    ]
    context_text = "\n\n".join(
        [
            f"[Sumber : {d.metadata.get('source_file')}, Hal : {d.metadata.get('page')}]\n{d.page_content}"
            for d in docs
        ]
    )

    final_prompt = prompt.format(
        context=context_text,
        question=question
    )
    print("="*50)
    print(context_text[:3000])
    print("="*50)
    answer = llm.invoke(final_prompt)
    latency = (
        time.perf_counter()
        - start_time
    )

    return {
        "answer": answer.content,
        "contexts": contexts,
        "latency": latency,
        "docs": docs
    }

In [35]:
results = []

for _, row in df.iterrows():

    result = banaspati_answer_eval(
        row["user_input"]
    )

    results.append({
        "question_id": row["question_id"],
        "question": row["user_input"],
        "reference": row["reference"],
        "expected_reference": row["expected_reference"],
        "answer": result["answer"],
        "contexts": result["contexts"],
        "latency": result["latency"]
    })

eval_df = pd.DataFrame(results)

eval_df.head()

[Sumber : Kurikulum.pdf, Hal : 381]
nar on Intelligent Technology and Its Applications (pp.
151-156).
Surabaya: ITS.
Dosen Pengampu 
Lecturers 
Henning Titi Ciptaningtyas, S.Kom., M.Kom.
Mata Kuliah Syarat 
Prerequisites 
Teknologi Komputasi Awan 
Cloud Computing Technology 
 
Mg 
Ke- 
Week 
Kemampuan akhir 
tiap tahapan belajar 
(Sub-CPMK) 
Final ability of each 
learning stage 
Penilaian 
Assessment 
Bentuk Pembelajaran, 
Metode Pembelajaran, 
Penugasan Mahasiswa, 
[ Estimasi Waktu] 
Form of Learning Method; 
Materi Pembelajaran 
[ Pustaka ] 
Learning Material 
[References] 
Bobot 
Penilai
an (%) 
Weigh
t

[Sumber : Kurikulum.pdf, Hal : 393]
| melakukan
optimasi
penjadwalan
tugas pada
komputasi awan
40%
completeness in
optimizing task
scheduling in
cloud computing | Rubrik penilaian
project
Project
assessment
rubric
Bentuk Penilaian
Tes: demo
project
Test: demo
project | Presentasi progres project
Presentation of project progress
Metode Pembelajaran
• Student talk [150’]
BM
Bentuk Pe

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.0-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 59.740116516s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url

In [9]:
for i, ref in enumerate(df["expected_reference"]):
    print(i+1, ref)

1 Jadwal Perkuliahan.docx; image1.png; bagian Rabu/Wednesday; baris 13.00–15.30; kolom TW2-705 untuk Teknologi Komputasi Awan A dan kolom TW2-702 untuk Kecerdasan Artifisial dan Machine Learning C.
2 Jadwal Perkuliahan.docx; image2.png; bagian Selasa/Tuesday; baris 13.00–15.30; kolom TW2-705 dan Lab 902.
3 Jadwal Perkuliahan.docx; image1.png; bagian Kamis/Thursday; kolom Lab 902; baris 07.30–10.00 dan 13.00–15.30.
4 Kurikulum.pdf; Tabel 17 Daftar Mata Kuliah Semester-IV; baris ET234405 Kecerdasan Artifisial dan Machine Learning. Jadwal Perkuliahan.docx; image2.png; bagian Selasa 13.00–15.30; kolom TW2-705 dan Lab 902.
5 Sosialisasi Magang dan Prestasi DTI.pdf; slide/halaman 20, 21, dan 23; bagian ketentuan magang DUDI, Magang Konversi SKS, onboarding, PKS, MK reguler, dan SIM Magang.
6 Sosialisasi Magang dan Prestasi DTI.pdf; slide/halaman 39, 42, 43, dan 45; bagian Magang Mandiri, deadline Surat Pengantar, dokumen konfirmasi magang, deadline PKS, dan konsekuensi PKS belum selesai.
7 K

In [14]:
import pandas as pd

chunks = pd.read_json(
    "../artifacts/chunks_recursive_1000_150.jsonl",
    lines=True
)
sources = chunks["metadata"].apply(
    lambda x: x.get("source_file", "")
)

for s in sorted(sources.unique()):
    print(s)

Data Dosen.pdf
Kalender-Akademik-ITS-Thn-Akademik-2025-2026.pdf
Kurikulum.pdf
Nilai snbt 2025.pdf
Peraturan Akademik.pdf
Sosialisasi Magang dan Prestasi DTI.pdf
Visi Misi Departemen.pdf


## Corpus Validation

Dataset source:
- 8 dokumen

Indexed corpus:

- 7 dokumen

In [31]:
import ragas

print(ragas.__version__)

0.4.3


NameError: name 'result_df' is not defined